In [0]:
%run ../delta_function

In [0]:
#bibliothèques à importer
import pandas as pd 
from pyspark.sql import functions as F
from pyspark.sql.functions import col, floor
from pyspark.sql import Window
from pyspark.sql.functions import current_timestamp
from pyspark.sql.types import TimestampType
import pyspark.sql.utils;
from pyspark.sql.types import StructType, StringType;
from pyspark.sql.functions import concat, lit, col, upper, max, when
from pyspark.sql.functions import udf
from pyspark.sql.types import IntegerType
from datetime import datetime, timedelta
from pyspark.sql.functions import regexp_replace
import numpy as np
from functools import reduce
from pyspark.sql import SparkSession
from pyspark.sql.functions import concat, lit, coalesce
from pyspark.sql.window import Window
from pyspark.sql.functions import col, expr
from pyspark.sql.types import FloatType
spark.conf.set("spark.sql.execution.arrow.enabled", "true")

In [0]:
try:
    verbose_mode = dbutils.widgets.get("verbose_mode");
except:
    verbose_mode = 'debug'
try:
    current_division = dbutils.widgets.get("division");
except:
    current_division = 'mal'
try:
    current_environment = dbutils.widgets.get("environment");
except:
    current_environment = 'dev' #dev
try:
    execution_mode = dbutils.widgets.get("execution_mode");
except:
    execution_mode = 'update'
try:
    current_project = dbutils.widgets.get("project");
except:
    current_project = 'maite_bi'
try:
    current_production_line = dbutils.widgets.get("production_line");
except:
    current_production_line = 'parameters'

current_catalog = current_division + '_' + current_project + '_' + current_environment;

current_schema = current_production_line;

current_location = 'abfss://' + current_project + '@adlsdpcom'+ current_environment +f'data.dfs.core.windows.net/' + current_production_line + '/'

if verbose_mode == 'debug':
    display("Debug Mode")
    display(f"current_division : {current_division}")
    display(f"current_environment : {current_environment}")
    display(f"current_project : {current_project}")
    display(f"current_production_line : {current_production_line}")
    display(f"current_catalog : {current_catalog}")
    display(f"current_schema : {current_schema}")
    display(f"current_location : {current_location}")

In [0]:
catalog_prd = f"""mal_maite_{current_environment}"""

In [0]:
nogent1_features = spark.table(f"{catalog_prd}.nogent1.features") 
nogent2_features = spark.table(f"{catalog_prd}.nogent2.features") 
rouen1_features = spark.table(f"{catalog_prd}.rouen1.features") 
strasbourg2_features = spark.table(f"{catalog_prd}.strasbourg2.features") 
prouvy1_features = spark.table(f"{catalog_prd}.prouvy1.features") 
polisy1_features = spark.table(f"{catalog_prd}.polisy1.features") 
buzau1_features = spark.table(f"{catalog_prd}.buzau1.features")
bolelemi1_features = spark.table(f"{catalog_prd}.bolelemi1.features")

In [0]:
df_features = nogent1_features.unionByName(nogent2_features).unionByName(rouen1_features).unionByName(strasbourg2_features).unionByName(prouvy1_features).unionByName(polisy1_features).unionByName(buzau1_features).unionByName(bolelemi1_features)

df_features_filtered = df_features.filter(
    (F.col("is_controllable") == True)
    & (F.col("deleted") == False)
)

display(df_features_filtered)

In [0]:
# Colonnes à convertir de decimal -> float
decimal_cols = ["min_value", "max_value", "step"]

# Conversion
df_features_converted = df_features_filtered.select(
    *[
        col(c).cast(FloatType()).alias(c) if c in decimal_cols else col(c)
        for c in df_features_filtered.columns
    ]
)

## Bornes dynamiques

`min_value` / `max_value` ne sont plus repris tels quels des tables `features` : ils sont
recalcules sur la population de reference, avec **le code du projet d'inference copie tel quel**
(DM-6232 / DM-6452), pour que le rapport affiche les bornes que l'optimiseur utilise vraiment.

Population de reference : `deleted = false`, `batch_status` dans ('finished', 'in_progress'),
moins les `batch_id` listes dans `mttts_ignore`. Puis percentile 1 / percentile 99, arrondi,
et calage sur la grille du `step`.

Le calcul tourne en pandas et non en Spark : c'est ce qui permet d'executer leur code sans le
retranscrire. La lecture des tables reste en Spark, seul acces aux tables Delta.

Les valeurs statiques des tables `features` restent le repli quand la feature est absente de la
master table ou n'y a aucune donnee valide -- leur fonction s'en charge elle-meme.

In [ ]:
# Memes sites que les tables features chargees plus haut.
sites = ["nogent1", "nogent2", "rouen1", "strasbourg2", "prouvy1", "polisy1", "buzau1", "bolelemi1"]

In [ ]:
import hashlib

# derive_bounds() n'est pas appelee mais recopiee plus bas : rien ne garantit donc que
# notre copie reste fidele si la DS fait evoluer son calcul. On garde l'empreinte du
# fichier source et on s'arrete des qu'elle bouge -- une copie perimee ne leverait
# aucune erreur, elle afficherait juste des bornes que l'optimiseur n'utilise plus.
#
# Au premier run, la cellule affiche l'empreinte a reporter ici.
BOUNDS_PY_SHA256 = ""

bounds_py = f"/Workspace/Shared/maite/{sites[0]}/mal_maite_code/files/inference/src/optimizers/bounds.py"

try:
    current_sha = hashlib.sha256(open(bounds_py, "rb").read()).hexdigest()
except FileNotFoundError:
    current_sha = None
    print(f"ATTENTION : {bounds_py} introuvable, la derive ne peut pas etre verifiee")

if current_sha and not BOUNDS_PY_SHA256:
    print(f"Empreinte a reporter dans BOUNDS_PY_SHA256 : {current_sha}")
elif current_sha and current_sha != BOUNDS_PY_SHA256:
    raise RuntimeError(
        f"bounds.py a change cote inference (empreinte {current_sha}).\n"
        f"Comparer derive_bounds / round_bound / snap_to_step avec la copie de ce notebook, "
        f"reporter les evolutions, puis mettre a jour BOUNDS_PY_SHA256."
    )
else:
    print(f"bounds.py inchange depuis la recopie ({bounds_py})")

In [ ]:
# ==============================================================================
# Code repris tel quel du projet d'inference -- NE RIEN REECRIRE ICI.
#
#   files/inference/src/optimizers/bounds.py
#   files/inference/src/entities/recommendation_input_tables.py  (les deux filtres)
#
# Ces fonctions decident des bornes que l'optimiseur explore. Toute reformulation,
# meme equivalente en apparence, peut decaler les valeurs du rapport par rapport a
# celles utilisees en production, sans qu'aucune erreur ne le signale.
#
# Seule adaptation : le logger, le leur n'existant pas dans ce notebook.
# ==============================================================================

import math
from dataclasses import dataclass


class _Logger:
    def info(self, msg): print(f"INFO  {msg}")
    def warning(self, msg): print(f"WARN  {msg}")
    def error(self, msg): print(f"ERROR {msg}")


logger = _Logger()

ACTIVE_BATCH_STATUSES = ("finished", "in_progress")

BOUND_FOR_MIN_PCTL = 0.01
BOUND_FOR_MAX_PCTL = 0.99


@dataclass(frozen=True)
class FeatureBounds:
    min_val: float
    max_val: float


def round_bound(value):
    """
    Round a bound value to at most 2 decimal places.
    Returns an int when the rounded value is whole (e.g. 3.0 → 3).
    """
    rounded = round(float(value), 2)
    return int(rounded) if rounded == int(rounded) else rounded


def snap_to_step(min_val, max_val, step):
    """
    Snap bounds to the step grid: floor min DOWN and ceil max UP to the nearest
    multiple of step, privileging clean integer values.

    The 1e-9 rounding guards against floating-point drift before the floor/ceil
    so grid-aligned inputs are not nudged off-grid.
    """
    def _snap(value, fn):
        snapped = fn(round(value / step, 9)) * step
        return int(snapped) if snapped == int(snapped) else round(snapped, 9)

    return _snap(min_val, math.floor), _snap(max_val, math.ceil)


def filter_active_batches(mt):
    """
    Keep non-deleted batches that are finished or in progress: only these serve as a reference
    and shape the search space. In-progress batches are included so the search space reacts to
    current maltster behaviour.
    """
    missing_columns = [col for col in ("deleted", "batch_status") if col not in mt.columns]
    if missing_columns:
        msg = f"master_table has no {missing_columns} column(s) — cannot apply the batch status filter"
        logger.error(msg)
        raise ValueError(msg)
    return mt[(mt["deleted"] == False) & (mt["batch_status"].isin(ACTIVE_BATCH_STATUSES))]  # noqa: E712


def exclude_ignored_batches(mt, mttts_ignore):
    """
    Drop the batches listed in mttts_ignore.csv, the same batches training refuses to learn from.
    """
    if mttts_ignore.empty or "batch_id" not in mttts_ignore.columns:
        logger.info("mttts_ignore holds no batch_id — no batch excluded from the reference population")
        return mt, 0
    if "batch_id" not in mt.columns:
        msg = "master_table has no batch_id column — cannot apply the ignored batches exclusion"
        logger.error(msg)
        raise ValueError(msg)

    ignored_ids = set(mttts_ignore["batch_id"].astype(str))
    return mt[~mt["batch_id"].astype(str).isin(ignored_ids)], len(ignored_ids)


def derive_bounds(active_mt, features_to_optimize):
    """
    Derive min/max search-space bounds for every feature to optimize from
    active_mt in a single pass (percentile_01 for min, percentile_99 for max),
    rounded and snapped to each feature's step grid.

    Falls back to the static min_value/max_value from features_to_optimize
    when a feature column is missing from active_mt or has no valid
    (non-null) data.
    """
    present_features = [key for key in features_to_optimize if key in active_mt.columns]
    numeric_mt = active_mt[present_features].apply(pd.to_numeric, errors="coerce") if present_features else pd.DataFrame()
    quantiles = numeric_mt.quantile([BOUND_FOR_MIN_PCTL, BOUND_FOR_MAX_PCTL]) if present_features else pd.DataFrame()

    bounds = {}
    for key, val in features_to_optimize.items():
        fallback_min = val["min_value"]
        fallback_max = val["max_value"]

        if key not in present_features:
            logger.info(f"'{key}' not found in the reference population — falling back to static bounds "
                        f"[{fallback_min}, {fallback_max}] from features.csv")
            bounds[key] = FeatureBounds(fallback_min, fallback_max)
            continue

        n_rows = int(numeric_mt[key].notna().sum())

        if n_rows == 0:
            logger.info(f"'{key}' has no valid data in the reference population — falling back to static bounds "
                        f"[{fallback_min}, {fallback_max}] from features.csv")
            bounds[key] = FeatureBounds(fallback_min, fallback_max)
            continue

        min_val = round_bound(quantiles.loc[BOUND_FOR_MIN_PCTL, key])
        max_val = round_bound(quantiles.loc[BOUND_FOR_MAX_PCTL, key])

        step = val["step"]
        if step and step > 0:
            snapped_min, snapped_max = snap_to_step(min_val, max_val, step)
            if (snapped_min, snapped_max) != (min_val, max_val):
                logger.info(f"'{key}' — step coherence (step={step}): "
                            f"[{min_val}, {max_val}] → [{snapped_min}, {snapped_max}]")
            min_val, max_val = snapped_min, snapped_max

        bounds[key] = FeatureBounds(min_val, max_val)
        logger.info(f"Dynamic bounds for {key}: [{min_val}, {max_val}] (n={n_rows})")

    return bounds

In [ ]:
features_pd = df_features_converted.toPandas()
feature_cols = sorted(set(features_pd["feature_reference"]))
print(f"{len(feature_cols)} features controlables distinctes")


def load_reference_population(site):
    """
    Lit la master table et mttts_ignore du site, les passe en pandas, puis applique les deux
    filtres du projet d'inference.

    La lecture reste en Spark, seul acces aux tables Delta, mais le calcul bascule en pandas
    des que possible : c'est ce qui permet d'executer leur code tel quel au lieu de le
    retranscrire. Le volume le permet -- une ligne par batch -- et on ne rapatrie que les
    colonnes utiles plutot que les ~500 de la master table.
    """
    mt = spark.table(f"mal_maite_{site}_{current_environment}.gold.master_table")

    # Les colonnes de filtrage ne sont selectionnees que si elles existent : leur absence doit
    # remonter depuis filter_active_batches, avec son message, plutot que d'echouer ici sur un
    # nom de colonne inconnu.
    filter_cols = [c for c in ("deleted", "batch_status", "batch_id", "production_line") if c in mt.columns]
    present_features = [f for f in feature_cols if f in mt.columns]

    mt_pd = mt.select(*filter_cols, *present_features).toPandas()
    ignore_pd = spark.table(f"{catalog_prd}.{site}.mttts_ignore").toPandas()

    reference = filter_active_batches(mt_pd)
    n_active = len(reference)
    reference, n_ignored = exclude_ignored_batches(reference, ignore_pd)
    reference = reference.reset_index(drop=True)
    n_removed = n_active - len(reference)

    logger.info(
        f"{site} - Reference population: {len(mt_pd)} rows "
        f"-> {n_active} after status filter {ACTIVE_BATCH_STATUSES} "
        f"-> {len(reference)} after excluding {n_removed} row(s) "
        f"matching {n_ignored} ignored batch_id(s)"
    )

    # Un ecart de format entre les batch_id des deux tables ne leve aucune erreur : la liste
    # d'exclusion devient juste sans effet, et les batchs ecartes par le metier reviennent
    # peser sur les bornes.
    if n_ignored and not n_removed:
        logger.warning(
            f"{site} - None of the {n_ignored} ignored batch_id(s) matched a batch of the master "
            f"table: the ignore list is having no effect."
        )

    return reference

In [ ]:
rows = []

for site in sites:
    reference = load_reference_population(site)

    # production_line est en minuscules dans la master table et en majuscules dans les tables
    # features : sans cette normalisation aucune ligne de production ne serait reconnue et
    # toutes les bornes repartiraient sur leurs valeurs statiques, sans erreur.
    reference["production_line"] = reference["production_line"].str.upper()

    # Groupe par production_line plutot que de supposer une ligne par site : le regroupement
    # est correct dans les deux cas, l'hypothese ne l'est pas forcement.
    for production_line, group in reference.groupby("production_line"):
        line_features = features_pd[features_pd["production_line"] == production_line]
        if line_features.empty:
            logger.warning(f"{site} - {production_line} absente des tables features, ignoree")
            continue

        features_to_optimize = {
            r.feature_reference: {"min_value": r.min_value, "max_value": r.max_value, "step": r.step}
            for r in line_features.itertuples()
        }

        for feature, bound in derive_bounds(group, features_to_optimize).items():
            rows.append((production_line, feature, bound.min_val, bound.max_val))

dynamic_bounds = pd.DataFrame(rows, columns=["production_line", "feature_reference", "new_min", "new_max"])
print(f"{len(dynamic_bounds)} bornes resolues par derive_bounds")

In [ ]:
merged = features_pd.merge(dynamic_bounds, on=["production_line", "feature_reference"], how="left")

# derive_bounds rend toujours une borne, calculee ou de repli. Une valeur absente ici signale
# donc autre chose : un couple (ligne de production, feature) qui n'a jamais atteint la fonction,
# faute de population de reference pour cette ligne.
n_unresolved = int(merged["new_min"].isna().sum())
n_changed = int(
    (merged["new_min"].notna() & (merged["new_min"] != merged["min_value"])).sum()
    + (merged["new_max"].notna() & (merged["new_max"] != merged["max_value"])).sum()
)

merged["min_value"] = merged["new_min"].fillna(merged["min_value"]).astype(float)
merged["max_value"] = merged["new_max"].fillna(merged["max_value"]).astype(float)

print(f"{len(merged) - n_unresolved} / {len(merged)} lignes resolues, "
      f"{n_unresolved} sans population de reference (valeurs actuelles conservees)")
print(f"{n_changed} borne(s) modifiee(s) par rapport aux valeurs actuelles")

df_features_with_dynamic_bounds = spark.createDataFrame(merged[df_features_converted.columns])
for c in decimal_cols:
    df_features_with_dynamic_bounds = df_features_with_dynamic_bounds.withColumn(
        c, F.col(c).cast(FloatType())
    )

IMPORT

In [0]:
table_features_min_max = df_features_with_dynamic_bounds.select(
    "production_line",
    "feature_reference",
    "min_value",
    "max_value",
    "is_controllable",
    "deleted",
    "is_categorical",
    "is_stratification",
    "step",
    "activity"
)

In [0]:
current_process= "fact_features_min_max"

In [0]:
target_fact_features_min_max = current_catalog +"."+current_schema+"."+current_process
print(target_fact_features_min_max)

In [0]:
all_columns =  table_features_min_max.columns
display(all_columns)

In [0]:

# define the primary key 
primary_key = [    
    'production_line'
    ,'feature_reference']

additional_columns = get_additional_columns(all_columns, primary_key)

if verbose_mode == 'debug': 
    print(additional_columns)

In [0]:
handle_table_update(
    table_features_min_max, 
    target_fact_features_min_max, 
    primary_key, 
    all_columns,
    additional_columns_to_check=additional_columns,
    mode=execution_mode # Use "update" for update mode, "full" for delete/insert mode
    )